In [ ]:
from astropy.io import fits
import glob
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset

In [ ]:
# # Función para graficar un espectro aleatorio
# def plot_random_spectrum(flux, wavelength, title=""):
#     idx = np.random.randint(0, flux.shape[0])
#     plt.figure(figsize=(8,4))
#     plt.plot(wavelength[idx], flux[idx])
#     plt.xlabel("Wavelength")
#     plt.ylabel("Flux")
#     plt.title(f"{title} - Spectrum index {idx}")
#     plt.show()

# # Función para cargar datos usando np.memmap y copiarlos a memoria
# def load_data(file_flux, file_wave, file_redshift, total, num_points):
#     flux = np.memmap(file_flux, dtype="float32", mode="r", shape=(total, num_points))
#     wavelength = np.memmap(file_wave, dtype="float32", mode="r", shape=(total, num_points))
#     redshift = np.memmap(file_redshift, dtype="float32", mode="r", shape=(total,))
#     return np.array(flux), np.array(wavelength), np.array(redshift)

# # Parámetros generales
# data_dir = "data"
# num_points = 5000
# big_total = 595472    # Número de espectros en bigtraining
# small_total = 4528    # Número de espectros en smalltraining

# # Rutas de archivos para bigtraining (archivos sin normalizar)
# big_flux_path     = os.path.join(data_dir, "spectra_data_bigtraining_flux.dat")
# big_wave_path     = os.path.join(data_dir, "spectra_data_bigtraining_wavelength.dat")
# big_redshift_path = os.path.join(data_dir, "spectra_data_bigtraining_redshift.dat")

# # Rutas de archivos para smalltraining
# small_flux_path     = os.path.join(data_dir, "spectra_data_smalltraining_flux.dat")
# small_wave_path     = os.path.join(data_dir, "spectra_data_smalltraining_wavelength.dat")
# small_redshift_path = os.path.join(data_dir, "spectra_data_smalltraining_redshift.dat")

# # Cargar datos
# print("Cargando datos de bigtraining...")
# flux_big, wavelength_big, redshift_big = load_data(big_flux_path, big_wave_path, big_redshift_path, big_total, num_points)
# print("Cargando datos de smalltraining...")
# flux_small, wavelength_small, redshift_small = load_data(small_flux_path, small_wave_path, small_redshift_path, small_total, num_points)

# # Mostrar un espectro aleatorio de cada dataset (Raw)
# plot_random_spectrum(flux_big, wavelength_big, "Bigtraining (Raw)")
# plot_random_spectrum(flux_small, wavelength_small, "Smalltraining (Raw)")

# # Dividir bigtraining en train (84%) y test (16%)
# indices_big = np.arange(big_total)
# big_train_indices, big_test_indices = train_test_split(indices_big, test_size=0.16, random_state=42)

# flux_big_train      = flux_big[big_train_indices]
# wavelength_big_train = wavelength_big[big_train_indices]
# redshift_big_train   = redshift_big[big_train_indices]

# flux_big_test       = flux_big[big_test_indices]
# wavelength_big_test  = wavelength_big[big_test_indices]
# redshift_big_test    = redshift_big[big_test_indices]

# plot_random_spectrum(flux_big_train, wavelength_big_train, "Bigtraining Train (Raw)")
# plot_random_spectrum(flux_big_test, wavelength_big_test, "Bigtraining Test (Raw)")

# # Crear el conjunto de entrenamiento final: bigtraining_train + smalltraining
# flux_train = np.concatenate([flux_big_train, flux_small], axis=0)
# wavelength_train = np.concatenate([wavelength_big_train, wavelength_small], axis=0)
# redshift_train = np.concatenate([redshift_big_train, redshift_small], axis=0)

# plot_random_spectrum(flux_train, wavelength_train, "Conjunto de Entrenamiento Final (Raw)")

# # Convertir a float32 para reducir uso de memoria
# flux_train = flux_train.astype(np.float32)
# wavelength_train = wavelength_train.astype(np.float32)
# flux_big_test = flux_big_test.astype(np.float32)
# wavelength_big_test = wavelength_big_test.astype(np.float32)

# # Definir variables para el escalado incremental
# chunk_size = 10000  # Puedes ajustar este valor según la memoria disponible
# num_samples = flux_train.shape[0]
# print(f"Total de muestras en entrenamiento: {num_samples}")

# # Ajuste incremental de StandardScaler para flux y wavelength
# flux_scaler = StandardScaler()
# wavelength_scaler = StandardScaler()
# print("Ajustando escaladores en chunks...")
# for start in range(0, num_samples, chunk_size):
#     end = min(start + chunk_size, num_samples)
#     flux_scaler.partial_fit(flux_train[start:end])
#     wavelength_scaler.partial_fit(wavelength_train[start:end])

# # Transformar el conjunto de entrenamiento en chunks usando np.memmap para evitar asignación masiva en RAM
# flux_train_scaled = np.memmap("flux_train_scaled.dat", dtype="float32", mode="w+", shape=flux_train.shape)
# wavelength_train_scaled = np.memmap("wavelength_train_scaled.dat", dtype="float32", mode="w+", shape=wavelength_train.shape)
# print("Transformando conjunto de entrenamiento en chunks...")
# for start in range(0, num_samples, chunk_size):
#     end = min(start + chunk_size, num_samples)
#     flux_train_scaled[start:end] = flux_scaler.transform(flux_train[start:end])
#     wavelength_train_scaled[start:end] = wavelength_scaler.transform(wavelength_train[start:end])

# # Transformar el conjunto de test en chunks usando np.memmap
# num_samples_test = flux_big_test.shape[0]
# flux_test_scaled = np.memmap("flux_test_scaled.dat", dtype="float32", mode="w+", shape=flux_big_test.shape)
# wavelength_test_scaled = np.memmap("wavelength_test_scaled.dat", dtype="float32", mode="w+", shape=wavelength_big_test.shape)
# print("Transformando conjunto de test en chunks...")
# for start in range(0, num_samples_test, chunk_size):
#     end = min(start + chunk_size, num_samples_test)
#     flux_test_scaled[start:end] = flux_scaler.transform(flux_big_test[start:end])
#     wavelength_test_scaled[start:end] = wavelength_scaler.transform(wavelength_big_test[start:end])

# # Mostrar un espectro aleatorio del conjunto de entrenamiento escalado
# plot_random_spectrum(flux_train_scaled, wavelength_train_scaled, "Conjunto de Entrenamiento Final (Scaled)")

# # Convertir a tensores de PyTorch (cargando el memmap en array)
# train_data = {
#     "flux": torch.tensor(np.array(flux_train_scaled), dtype=torch.float32),
#     "wavelength": torch.tensor(np.array(wavelength_train_scaled), dtype=torch.float32),
#     "redshift": torch.tensor(redshift_train, dtype=torch.float32)
# }
# test_data = {
#     "flux": torch.tensor(np.array(flux_test_scaled), dtype=torch.float32),
#     "wavelength": torch.tensor(np.array(wavelength_test_scaled), dtype=torch.float32),
#     "redshift": torch.tensor(redshift_big_test, dtype=torch.float32)
# }

# # Guardar los datasets preprocesados
# torch.save(train_data, os.path.join(data_dir, "train_dataset.pt"))
# torch.save(test_data, os.path.join(data_dir, "test_dataset.pt"))
# print("Datasets guardados exitosamente.")

# # Guardar los escaladores ajustados en un diccionario y luego en un archivo pickle
# scaler_dict = {
#     "flux_scaler": flux_scaler,
#     "wavelength_scaler": wavelength_scaler
# }
# scaler_filename = os.path.join(data_dir, "scaler_modelCNN_UPD_fitted.pkl")
# with open(scaler_filename, "wb") as f:
#     pickle.dump(scaler_dict, f)
# print(f"Scalers guardados en {scaler_filename}")

# # Convertir a tensores de PyTorch sin convertir primero a array completo
# train_data = {
#     "flux": torch.from_numpy(flux_train_scaled),           # No se copia el contenido completo
#     "wavelength": torch.from_numpy(wavelength_train_scaled),
#     "redshift": torch.from_numpy(redshift_train.astype(np.float32))
# }
# test_data = {
#     "flux": torch.from_numpy(flux_test_scaled),
#     "wavelength": torch.from_numpy(wavelength_test_scaled),
#     "redshift": torch.from_numpy(redshift_big_test.astype(np.float32))
# }

# # Guardar los datasets preprocesados
# torch.save(train_data, os.path.join(data_dir, "train_dataset.pt"))
# torch.save(test_data, os.path.join(data_dir, "test_dataset.pt"))
# print("Datasets guardados exitosamente.")

# # Guardar los escaladores ajustados en un diccionario y luego en un archivo pickle
# scaler_dict = {
#     "flux_scaler": flux_scaler,
#     "wavelength_scaler": wavelength_scaler
# }
# scaler_filename = os.path.join(data_dir, "scaler_modelCNN_UPD_fitted.pkl")
# with open(scaler_filename, "wb") as f:
#     pickle.dump(scaler_dict, f)
# print(f"Scalers guardados en {scaler_filename}")

Datasets guardados exitosamente.
Scalers guardados en data\scaler_modelCNN_UPD_fitted.pkl


In [ ]:
# # Definir el Dataset
# class SpectraDataset(Dataset):
#     def __init__(self, flux_path, wavelength_path, redshift_path, total, num_points):
#         """
#         Parámetros:
#           flux_path, wavelength_path, redshift_path: rutas a los archivos .dat (ya normalizados para flujo y longitud de onda)
#           total: número total de espectros
#           num_points: número de puntos por espectro
#         """
#         self.flux = np.memmap(flux_path, dtype="float32", mode="r", shape=(total, num_points))
#         self.wavelength = np.memmap(wavelength_path, dtype="float32", mode="r", shape=(total, num_points))
#         self.redshift = np.memmap(redshift_path, dtype="float32", mode="r", shape=(total,))
#         self.total = total
#         self.num_points = num_points

#     def __len__(self):
#         return self.total

#     def __getitem__(self, idx):
#         # Leer la muestra individualmente
#         flux_sample = self.flux[idx, :].copy()
#         wave_sample = self.wavelength[idx, :].copy()
#         # Combinar en un array de forma (2, num_points)
#         X = np.stack([flux_sample, wave_sample], axis=0)
#         # Leer el redshift
#         y = self.redshift[idx]
#         # Convertir a tensores de PyTorch
#         X_tensor = torch.tensor(X, dtype=torch.float32)
#         # Se aplica unsqueeze para que el target tenga forma (1,)
#         y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(0)
#         return X_tensor, y_tensor

# # Parámetros y rutas
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Usando dispositivo: {device}")

# data_dir = "data"
# num_points = 5000

# # Parámetros de tamaño
# big_total = 595472     # Número de espectros en bigtraining
# small_total = 4528     # Número de espectros en smalltraining

# # Rutas de los archivos bigtraining (datos normalizados)
# big_flux_norm_path = os.path.join(data_dir, "spectra_data_bigtraining_flux_norm.dat")
# big_wavelength_norm_path = os.path.join(data_dir, "spectra_data_bigtraining_wavelength_norm.dat")
# big_redshift_path = os.path.join(data_dir, "spectra_data_bigtraining_redshift_norm.dat")

# # Rutas de los archivos smalltraining (datos normalizados)
# small_flux_norm_path = os.path.join(data_dir, "spectra_data_smalltraining_flux_norm.dat")
# small_wavelength_norm_path = os.path.join(data_dir, "spectra_data_smalltraining_wavelength_norm.dat")
# small_redshift_path = os.path.join(data_dir, "spectra_data_smalltraining_redshift_norm.dat")

# # Dividir el conjunto bigtraining en train y test (84% train, 16% test)
# indices_big = np.arange(big_total)
# big_train_indices, big_test_indices = train_test_split(indices_big, test_size=0.16, random_state=42)

# # Crear los Datasets con los datos ya normalizados
# big_dataset_norm = SpectraDataset(big_flux_norm_path, big_wavelength_norm_path, big_redshift_path, big_total, num_points)
# small_dataset_norm = SpectraDataset(small_flux_norm_path, small_wavelength_norm_path, small_redshift_path, small_total, num_points)

# # Aplicar los mismos índices para los subconjuntos de bigtraining
# big_train_dataset_norm = Subset(big_dataset_norm, big_train_indices)
# big_test_dataset_norm = Subset(big_dataset_norm, big_test_indices)

# # Conjunto de entrenamiento final normalizado: 84% de bigtraining + todo smalltraining
# train_dataset = ConcatDataset([big_train_dataset_norm, small_dataset_norm])
# test_dataset = big_test_dataset_norm  # El test proviene solo de bigtraining

# # Crear DataLoaders (puedes ajustar batch_size y num_workers según convenga)
# train_loader = DataLoader(train_dataset, batch_size=8192, shuffle=True, num_workers=0, pin_memory=True)
# test_loader = DataLoader(test_dataset, batch_size=8192, shuffle=False, num_workers=0, pin_memory=True)

# print("Listo para entrenar a partir de los datos normalizados y usando memmap de forma eficiente.")

In [ ]:
# Definir la clase Dataset
class SpectraDataset(Dataset):
    def __init__(self, dataset_path):
        # Cargamos el diccionario previamente guardado (.pt)
        data = torch.load(dataset_path)
        self.flux = data["flux"]          # Tensor con forma [N, num_points]
        self.wavelength = data["wavelength"]  # Tensor con forma [N, num_points]
        self.redshift = data["redshift"]      # Tensor con forma [N]
    
    def __len__(self):
        return self.flux.shape[0]
    
    def __getitem__(self, idx):
        input_sample = torch.stack([self.flux[idx], self.wavelength[idx]], dim=0)  # [2, num_points]
        target = self.redshift[idx]
        return input_sample, target

# Rutas a los archivos preprocesados
data_dir = "data"
train_dataset_path = os.path.join(data_dir, "train_dataset.pt")
test_dataset_path  = os.path.join(data_dir, "test_dataset.pt")

# Crear los datasets
train_dataset = SpectraDataset(train_dataset_path)
test_dataset  = SpectraDataset(test_dataset_path)

# Crear DataLoaders
batch_size = 32  # Ajusta el tamaño del batch según tu GPU/CPU
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

In [ ]:
# Configurar dispositivo para GPU si está disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

# Número de puntos en cada espectro
num_points = 5000

# Definir el modelo CNN en PyTorch con dropout para regularización
class CNN(nn.Module):
    def __init__(self, num_points):
        super(CNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv1d(in_channels=2, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Conv1d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2)
        )
        # Tras 3 max pooling, la dimensión se reduce en un factor de 8
        conv_output_size = num_points // 8
        self.fc_layers = nn.Sequential(
            nn.Linear(64 * conv_output_size, 128),
            nn.ReLU(),
            nn.Dropout(p=0.5),  # Regularización con Dropout
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

modelCNN = CNN(num_points).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(modelCNN.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

In [ ]:
# Entrenar el modelo
num_epochs = 5
for epoch in range(num_epochs):
    modelCNN.train()
    running_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = modelCNN(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_X.size(0)
    epoch_loss = running_loss / len(train_dataset)
    
    # Evaluación en el conjunto de validación
    modelCNN.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = modelCNN(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item() * batch_X.size(0)
    val_loss /= len(test_dataset)
    
    # Actualizar la tasa de aprendizaje según la pérdida de validación
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, Val Loss: {val_loss:.4f}, LR: {current_lr:.10f}", flush=True)

# Evaluación final en el conjunto de prueba utilizando MAE
mae_loss = nn.L1Loss()
modelCNN.eval()
test_mae = 0.0
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        outputs = modelCNN(batch_X)
        loss = mae_loss(outputs, batch_y)
        test_mae += loss.item() * batch_X.size(0)
test_mae /= len(test_dataset)
print(f"Error absoluto medio en el conjunto de prueba: {test_mae:.4f}")

# Guardar los parámetros del modelo
checkpoint = {
    'epoch': num_epochs,
    'model_state_dict': modelCNN.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict()
}
torch.save(checkpoint, 'storage/modelCNN_UPD_600ktest.pth')

In [ ]:
# Seguir entrenando el modelo
checkpoint = torch.load('storage/modelCNN_UPD_600ktest.pth', weights_only=False)
modelCNN.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
start_epoch = checkpoint['epoch']

# Definir el número total de epochs que deseas entrenar
num_epochs = 2

# Continuar el entrenamiento desde el epoch donde se quedó
for epoch in range(start_epoch, num_epochs):
    modelCNN.train()
    running_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = modelCNN(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_X.size(0)
    epoch_loss = running_loss / len(train_dataset)

    # Evaluación en el conjunto de validación
    modelCNN.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = modelCNN(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item() * batch_X.size(0)
    val_loss /= len(test_dataset)

    # Actualizar la tasa de aprendizaje según la pérdida de validación
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    if (epoch + 1) % 1 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, Val Loss: {val_loss:.4f}, LR: {current_lr:.10f}")

# Evaluación final en el conjunto de prueba utilizando MAE
mae_loss = nn.L1Loss()
modelCNN.eval()
test_mae = 0.0
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        outputs = modelCNN(batch_X)
        loss = mae_loss(outputs, batch_y)
        test_mae += loss.item() * batch_X.size(0)
test_mae /= len(test_dataset)
print(f"Error absoluto medio en el conjunto de prueba: {test_mae:.4f}")

# Guardar los parámetros del modelo y el scaler
checkpoint = {
    'epoch': num_epochs,
    'model_state_dict': modelCNN.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict()
}
torch.save(checkpoint, 'storage/modelCNN_UPD_600ktest.pth')

In [ ]:
# Obtener la lista inicial de archivos FITS
folder_path = r'spectrums'
files = [f for f in os.listdir(folder_path) if f.endswith('.fits')]
files = random.sample(files, len(files))
file_path = os.path.join(folder_path, files[0])

checkpoint = torch.load('storage/modelCNN_UPD_600ktest.pth', weights_only=False)
modelCNN.load_state_dict(checkpoint['model_state_dict'])
with open('extra/modelCNN_UPD_600ktest.pkl', 'rb') as f:
    scaler = pickle.load(f)


with fits.open(file_path) as hdul:
    test_flux = hdul[1].data["flux"]
    test_loglam = hdul[1].data["loglam"]
    test_redshift = hdul[2].data["Z"][0]  # Asumiendo que Z es un array y queremos el primer valor

test_wavelength = 10 ** test_loglam

def expand_points(wavelength, flux, target_count=5000):
    # Convertir a listas para facilitar las inserciones
    wl = list(wavelength)
    fl = list(flux)
    
    # Calcular las diferencias absolutas entre puntos consecutivos
    diffs = [abs(fl[i+1] - fl[i]) for i in range(len(fl)-1)]
    # Obtener los índices ordenados de mayor a menor diferencia
    sorted_indices = sorted(range(len(diffs)), key=lambda i: diffs[i], reverse=True)
    
    # Insertar nuevos puntos utilizando los índices ordenados
    while len(wl) < target_count:
        # Se recorre la lista de índices en orden descendente para evitar problemas con el reordenamiento
        for idx in sorted_indices:
            if len(wl) >= target_count:
                break
            # Calcular la interpolación lineal entre el punto idx y el siguiente
            new_wl = (wl[idx] + wl[idx+1]) / 2
            new_fl = (fl[idx] + fl[idx+1]) / 2
            # Insertar el nuevo punto en la posición correspondiente
            wl.insert(idx+1, new_wl)
            fl.insert(idx+1, new_fl)
    
    return np.array(wl), np.array(fl)

test_wavelength, test_flux = expand_points(test_wavelength, test_flux, target_count=num_points)

# Preprocesamiento para el modelo
# Canal 0: flux; Canal 1: wavelength
input_data = np.stack([test_flux, test_wavelength], axis=0)  # (2, 5000) 2 canales de tamaño 5000
input_data = input_data.reshape(1, 2, num_points)            # (1, 2, 5000) 1 muestra, 2 canales, 5000 puntos/canal

# Normalización
nsamples, nchannels, npoints = input_data.shape # Guardar dimensionalidad inicial
input_flat = input_data.reshape(nsamples, -1) # Aplanar/concatenar (1, 10000)
input_scaled = scaler.transform(input_flat) # Normalizar
input_scaled = input_scaled.reshape(nsamples, nchannels, npoints) # Recuperar dimensionalidad incial

# Convertir a tensor
input_tensor = torch.tensor(input_scaled, dtype=torch.float32)
input_tensor = input_tensor.to(device)

# Evaluar el modelo
with torch.no_grad():
    predicted_redshift = modelCNN(input_tensor)

print("Redshift real:", test_redshift)
print("Redshift predicho:", predicted_redshift.item())

plt.figure(figsize=(12, 6))
plt.plot(test_wavelength, test_flux, label="Test Espectro")
plt.xlabel("Longitud de onda (Ångstrom)")
plt.ylabel("Flujo (10^-17 erg/s/cm²/Å)")
plt.title("Espectro vs. Flujo (Test)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()